<div align="right"><a href="S17-Remez-design-Chebyshev-approximation.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). Beachte bitte, dass zur Ausführung von MATLAB in Notebooks zunächst die [MATLAB API für Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) installiert werden muss. Die Installation wird [hier](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html) beschrieben. <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/L17-Remez-Entwurf-%28Tschebyscheff-Approximation%29.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L16 Entwurf nichtrekursiver Filter mittels Fensterung](./L16-Entwurf-nichtrekursiver-Filter-mittels-Fensterung.ipynb) | [Aufgabe 17](./A17-Remez-Entwurf-%28Tschebyscheff-Approximation%29.ipynb) | [Inhalt / Übersicht](./00-MATLAB-Notebooks-Overview-DE.ipynb) | Weiter: [L19 Entwurf eines Hilbert-Transformators (Remez-Verfahren)](./L19-Entwurf-eines-Hilbert-Transformators-%28Remez-Verfahren%29.ipynb)

</div>

---


# Remez-Design (Chebyshev-Approximation)

Beim Entwurf von IIR-Filtern unter Verwendung der Chebyshev-Approximation wurde gezeigt, dass diese Approximation einen großen Vorteil bietet, da sie das gesamte Toleranzschema ausnutzt. Eine solche Entwurfsmethode (Anwendung der Chebyshev-Approximation) für FIR-Systeme ist in MATLAB in der Funktion `remez()` (Parks–McClellan-Algorithmus) implementiert. Die Approximation erfolgt in den Pass- und Stop-Bändern.

Der Aufruf von `b=remez(m,F,G)` entwirft einen Filter der Ordnung \(m\), dessen Spezifikation durch die Vektoren `F` und `G` vorgegeben ist. `F` enthält Paare von Frequenzen (zwischen 0 und 1), die die Pass- oder Stop-Bänder festlegen, und `G` ist die Frequenzantwort (Amplitude), die den Frequenzen in `F` entspricht.

**Beispiel:**
```matlab
F = [0 0.3 0.5 1]
G = [1 1 0 0]

## Aufgaben

**a)**  
Berechnen Sie einen Filter mit der obigen Frequenzantwort. Experimentieren Sie mit verschiedenen Filterordnungen und zählen Sie die Anzahl der Extremwerte innerhalb des Approximationsintervalls.

**b)**  
Der Aufruf von ```b=remez(m,F,G,W)``` gewichtet die gewünschte Frequenzantwort, wobei Frequenzbänder mit niedrigerem Gewicht weniger genau approximiert werden als Frequenzbänder mit höherem Gewicht. Die Anzahl der Gewichte im Vektor `W` entspricht der Hälfte der Anzahl der Einträge in `F` bzw. `G` (für jedes Frequenzband ein Gewichtsfaktor).

Das Frequenzband zwischen 0.0 und 0.3 (Passband) muss die gewünschte Frequenzantwort nicht sehr genau approximieren. Für das Frequenzband zwischen 0.5 und 1 sollte ein höheres Gewicht verwendet werden. Berechnen Sie den Filter erneut und vergleichen Sie die Frequenzantwort mit dem Filter aus Aufgabe a).

**c)**  
Entwerfen Sie die Low-Pass – High-Pass-Kombination wie unten angegeben, wobei die Passbänder (um den Faktor 4) höher gewichtet werden sollen als die Stopbänder:

| Band        | Frequenzbereich |
|-------------|----------------|
| Passband 1  | 0 – 0.2        |
| Stopband 1  | 0.26 – 0.44    |
| Passband 2  | 0.5 – 0.7      |
| Stopband 2  | 0.76 – 1       |

Filterordnung $m$: 128

**Hinweis:** Die ursprüngliche Funktion `remez()` wurde in MATLAB 7 und später durch `firpm()` ersetzt. Das Grundprinzip bleibt dasselbe, prüfen Sie jedoch die Dokumentation auf mögliche Unterschiede in der Parametersyntax.

In [ ]:
# BEGIN SOLUTION
% #####  Teilaufgabe a: Filter entwerfen  ######################################
f   = [0 0.3 0.5 1.0];     % Frequenzen der Sperr- und Durchlassbereiche
g   = [1 1 0 0];           % Impulsantwort
v_m = [9 10];              % Vektor mit den Filterordnungen

for m=v_m
    b = firpm(m,f,g);         % Filterentwurf nach Remez
    [H W] = freqz(b,1,512);   % Berechnung des Frequenzgangs

    % # grafische Ausgabe # %
    string = sprintf('Gewuenschte/Aktuelle Antwort - Filterordnung: %d', m);
    figure;
    plot(f,g,W/pi,abs(H));
    line([0 0.3],[ 2-max(abs(H)) 2-max(abs(H))],'linestyle','--'); % Erzeugung
    line([0 0.5],[max(abs(H)) max(abs(H))],'linestyle','--');      % einer Linie
    line([0.5 1],[max(abs(H(400:length(H))))...                    % im Plot
        max(abs(H(400:length(H))))],'linestyle','--');
    line([0.3 0.3],[ 2-max(abs(H)) 0],'linestyle','--');
    line([0.5 0.5],[ max(abs(H)) max(abs(H(400:length(H))))],'linestyle','--');
    grid;
    xlabel('Omega/pi');
    ylabel('|H(exp(j*Omega))|');
    title(string);
end;

% #####  Teilaufgabe b: Frequenzgang wichten  ##################################
f   = [0 0.3 0.5 1.0];     % Frequenzen der Sperr- und Durchlassbereiche
g   = [1 1 0 0];           % Impulsantwort
w   = [ 0.5 2 ];           % Gewichtung des Durchlass- und Sperrbereichs
v_m = [9 10];              % Vektor mit den Filterordnungen

for m=v_m
    b = firpm(m,f,g,w);
    [H W] = freqz(b,1,512);
    
    % # grafische Ausgabe # %
    string = sprintf('Gewuenschte/Aktuelle Antwort - Filterordnung: %d', m);
    figure; plot(f,g,W/pi,abs(H));
    line([0 0.3],[ 2-max(abs(H)) 2-max(abs(H))],'linestyle','--');
    line([0 0.5],[max(abs(H)) max(abs(H))],'linestyle','--');
    line([0.5 1],[max(abs(H(400:length(H)))) max(abs(H(400:length(H))))],...
        'linestyle','--');
    line([0.3 0.3],[ 2-max(abs(H)) 0],'linestyle','--');
    line([0.5 0.5],[ max(abs(H)) max(abs(H(400:length(H))))],'linestyle','--');
    grid;
    xlabel('Omega/pi');
    ylabel('|H(exp(j*Omega))|');
    title(string);
end;

% #####  Teilaufgabe c: Tiefpass-Bandpass Kombination  #########################
f = [0 0.2 0.26 0.44 0.5 0.7 0.76 1]; % Vektor der Eckfrequenzen
g = [1  1   0    0    1   1   0   0]; % Vektor der Frequenzgaenge
w = [ 1       4         1       4];   % Vektor mit den Gewichten
m = 128;                              % Filterordnung
b = firpm(m,f,g,w); 
[H W] = freqz(b,1,512);

% # grafische Ausgabe # %
figure;
plot(W/pi,20*log10(abs(H)) );
grid; 
xlabel('Omega/pi');
ylabel('|H(exp(j*Omega))| in dB');
title('TP-BP-Kombination');
# END SOLUTION

# Lösung 5.3: Remez-Design (Chebyshev-Approximation)

## Teil a
Beide dargestellten Amplitudenantworten sind Beispiele für den Entwurf linearer Tiefpassfilter unter Verwendung der Chebyshev-Approximation im Passband und Stopband.  
Der Entwurf erfolgt mithilfe des **Remez-Designs** (in MATLAB 7 und später durch `firpm()` ersetzt), wobei die Intervalle `[0, 0.3]` und `[0.5, 1]` festgelegt sind.  
Die Filterordnungen sind `m = 9` und `m = 10`.

<img src="./pngs/ldsv5a03_a1.png" width="300px" style="display:inline-block; margin-right:10px;">
<img src="./pngs/ldsv5a03_a2.png" width="300px" style="display:inline-block;">

Beide Plots bestätigen die gleichmäßige Approximation im Passband und Stopband.  
Die Anzahl der Extremwerte, die das Toleranzschema erfüllen, einschließlich der Randwerte, ist `floor(m/2) + 2` für `m = 9, 10` in beiden Amplitudenantworten.

## Teil b
Wenn das Stopband stärker gewichtet wird als das Passband (z. B. Faktor 4), ergeben sich für die gleichen Randbedingungen wie in Teil a folgende Frequenzantworten:

<img src="./pngs/ldsv5a03_b1.png" width="300px" style="display:inline-block; margin-right:10px;">
<img src="./pngs/ldsv5a03_b2.png" width="300px" style="display:inline-block;">

Ein Vergleich mit Teil a verdeutlicht den Einfluss der Gewichtung auf Stopband und Passband.

## Teil c
Aufgrund der gegebenen Frequenzbänder, Frequenzantworten und Gewichtsfaktoren ist der Entwurf selektiver Systeme mit dem **Remez-/firpm-Design** problemlos möglich.  
Das folgende Beispiel zeigt den Entwurf einer Tiefpass–Bandpass-Kombination, wie in der Aufgabe spezifiziert:

<img src="./pngs/ldsv5a03_c1.png" width="300px">